# 04 Business Interpretation

Phase 4A translates the completed PD and ECL outputs into business findings, dashboard-ready summary tables, resume material, and interview talking points. This notebook does not retrain the PD model, change ECL assumptions, or build the Streamlit dashboard.

## Project Context

This portfolio project demonstrates a full starter workflow for credit risk analytics: data understanding, baseline PD modeling, simplified IFRS 9-style ECL calculation, and business interpretation.

## Business Objective

Convert row-level ECL results into concise portfolio insights that can support an interview discussion, a GitHub project README, and a future dashboard.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PREDICTIONS_DIR = OUTPUTS_DIR / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

## Load ECL Outputs

In [2]:
ecl_path = OUTPUTS_DIR / "ecl_results.csv"
scenario_path = PREDICTIONS_DIR / "ecl_scenario_summary.csv"

ecl = pd.read_csv(ecl_path, low_memory=False)
scenarios = pd.read_csv(scenario_path)

print(f"Loaded ECL results: {ecl_path}")
print(f"Loaded scenario summary: {scenario_path}")
print(f"ECL shape: {ecl.shape}")
display(ecl.head())
display(scenarios)

Loaded ECL results: /Users/rovs/Documents/New project 2/projects/credit-risk-ifrs9-ecl-engine/outputs/ecl_results.csv
Loaded scenario summary: /Users/rovs/Documents/New project 2/projects/credit-risk-ifrs9-ecl-engine/outputs/predictions/ecl_scenario_summary.csv
ECL shape: (50000, 23)


,row_id,default_flag,pd_score,pd_score_band,ead,lgd,ifrs9_stage,ecl,loan_amnt,installment,term,int_rate,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,purpose,dti,delinq_2yrs,revol_util,total_acc
0,0,0,0.446690,Medium,3600.0,0.35,Stage 2,562.829344,3600.0,123.03,36 months,13.99,C,C4,10+ years,MORTGAGE,55000.0,Not Verified,debt_consolidation,5.91,0.0,29.7,13.0
1,1,0,0.572757,High,24700.0,0.35,Stage 3,4951.484660,24700.0,820.28,36 months,11.99,C,C1,10+ years,MORTGAGE,65000.0,Not Verified,small_business,16.06,1.0,19.2,38.0
2,2,0,0.259497,Very Low,20000.0,0.35,Stage 2,1816.478853,20000.0,432.66,60 months,10.78,B,B4,10+ years,MORTGAGE,63000.0,Not Verified,home_improvement,10.78,0.0,56.2,18.0
3,3,0,0.453106,Medium,35000.0,0.35,Stage 2,5550.547589,35000.0,829.90,60 months,14.85,C,C5,10+ years,MORTGAGE,110000.0,Source Verified,debt_consolidation,17.06,0.0,11.6,17.0
4,4,0,0.691391,Very High,10400.0,0.35,Stage 3,2516.663501,10400.0,289.91,60 months,22.45,F,F1,3 years,MORTGAGE,104433.0,Source Verified,major_purchase,25.37,1.0,64.5,35.0


,scenario,pd_multiplier,lgd_multiplier,total_exposure,total_ecl,average_pd,average_lgd,ecl_rate
0,Base,1.00,1.0,750967975.0,1.462972e+08,0.457411,0.414735,0.194812
1,Mild stress,1.25,1.1,750967975.0,2.007463e+08,0.570941,0.456208,0.267317
2,Severe stress,1.50,1.2,750967975.0,2.558385e+08,0.670914,0.497682,0.340678


## Portfolio Overview

In [3]:
total_loans = len(ecl)
total_exposure = ecl["ead"].sum()
total_ecl = ecl["ecl"].sum()
ecl_rate = total_ecl / total_exposure
average_pd = ecl["pd_score"].mean()
average_lgd = ecl["lgd"].mean()

overview = pd.DataFrame([
    {
        "total_loans": total_loans,
        "total_exposure": total_exposure,
        "total_ecl": total_ecl,
        "ecl_rate": ecl_rate,
        "average_pd": average_pd,
        "average_lgd": average_lgd,
    }
])
display(overview)

,total_loans,total_exposure,total_ecl,ecl_rate,average_pd,average_lgd
0,50000,750967975.0,1.462972e+08,0.194812,0.457411,0.414735


## PD Model Summary

The PD input is a baseline logistic regression model from Phase 2. It is a benchmark model, not a production credit decision model. Phase 2 test metrics were ROC AUC 0.701, accuracy 0.632, precision 0.285, recall 0.651, and F1 0.396.

## ECL Methodology Summary

- EAD uses `loan_amnt`.
- LGD uses a simplified home-ownership rule.
- ECL is calculated as `pd_score x lgd x ead`.
- Stress scenarios apply PD and LGD multipliers, capped at 100%.

## IFRS 9-style Staging Interpretation

In [4]:
def summarize_group(df, group_col):
    summary = (
        df.groupby(group_col, dropna=False, observed=False)
        .agg(
            loan_count=("ecl", "size"),
            total_exposure=("ead", "sum"),
            total_ecl=("ecl", "sum"),
            average_pd=("pd_score", "mean"),
            average_lgd=("lgd", "mean"),
        )
        .reset_index()
        .rename(columns={group_col: "group_value"})
    )
    summary["ecl_rate"] = summary["total_ecl"] / summary["total_exposure"]
    return summary[["group_value", "loan_count", "total_exposure", "total_ecl", "ecl_rate", "average_pd", "average_lgd"]].sort_values("total_ecl", ascending=False)

stage_summary = summarize_group(ecl, "ifrs9_stage")
display(stage_summary)

,group_value,loan_count,total_exposure,total_ecl,ecl_rate,average_pd,average_lgd
2,Stage 3,24484,387692675.0,1.003205e+08,0.258763,0.599055,0.429338
1,Stage 2,20521,289438775.0,4.215737e+07,0.145652,0.366453,0.405594
0,Stage 1,4995,73836525.0,3.819344e+06,0.051727,0.136799,0.380711


## Risk Concentration Analysis

In [5]:
score_band_summary = summarize_group(ecl, "pd_score_band")
display(score_band_summary)

grade_summary = summarize_group(ecl, "grade") if "grade" in ecl.columns else pd.DataFrame()
purpose_summary = summarize_group(ecl, "purpose") if "purpose" in ecl.columns else pd.DataFrame()

if not grade_summary.empty:
    display(grade_summary)
if not purpose_summary.empty:
    display(purpose_summary)

,group_value,loan_count,total_exposure,total_ecl,ecl_rate,average_pd,average_lgd
3,Very High,10000,174690100.0,5.393467e+07,0.308745,0.711032,0.437755
0,High,10000,149307950.0,3.554727e+07,0.238080,0.566579,0.429200
2,Medium,10000,139436675.0,2.638814e+07,0.189248,0.466126,0.416075
1,Low,10000,141060900.0,1.978006e+07,0.140224,0.354889,0.401875
4,Very Low,10000,146472350.0,1.064711e+07,0.072690,0.188429,0.388770


,group_value,loan_count,total_exposure,total_ecl,ecl_rate,average_pd,average_lgd
2,C,14516,214796950.0,4.647833e+07,0.216383,0.520506,0.418762
1,B,15264,207625275.0,3.280572e+07,0.158004,0.384287,0.414583
3,D,6872,112884000.0,3.024279e+07,0.267910,0.640262,0.419529
4,E,3496,65966725.0,1.953955e+07,0.296203,0.711388,0.419708
0,A,8641,126460250.0,9.648993e+06,0.076301,0.188433,0.400949
5,F,989,19341100.0,6.222157e+06,0.321706,0.763425,0.423357
6,G,222,3893675.0,1.359707e+06,0.349209,0.806166,0.433333


,group_value,loan_count,total_exposure,total_ecl,ecl_rate,average_pd,average_lgd
2,debt_consolidation,28283,447346375.0,9.156214e+07,0.204678,0.481754,0.415297
1,credit_card,12653,198308425.0,3.403521e+07,0.171628,0.401739,0.416628
3,home_improvement,2977,41958100.0,6.865066e+06,0.163617,0.413691,0.372237
8,other,2759,26249125.0,5.352529e+06,0.203913,0.457418,0.425933
5,major_purchase,1048,13260725.0,2.964457e+06,0.223552,0.491890,0.426765
10,small_business,482,7130250.0,1.895770e+06,0.265877,0.598656,0.427075
6,medical,583,5055725.0,1.152132e+06,0.227887,0.511496,0.427273
0,car,452,4600250.0,7.735541e+05,0.168155,0.364621,0.428982
4,house,183,2728900.0,6.408737e+05,0.234847,0.535059,0.436885
7,moving,280,2346400.0,6.108520e+05,0.260336,0.545070,0.468393


## Scenario Analysis Interpretation

In [6]:
scenario_view = scenarios.copy()
scenario_view["ecl_increase_vs_base"] = scenario_view["total_ecl"] - scenario_view.loc[scenario_view["scenario"] == "Base", "total_ecl"].iloc[0]
scenario_view["ecl_increase_pct_vs_base"] = scenario_view["ecl_increase_vs_base"] / scenario_view.loc[scenario_view["scenario"] == "Base", "total_ecl"].iloc[0]
display(scenario_view)

,scenario,pd_multiplier,lgd_multiplier,total_exposure,total_ecl,average_pd,average_lgd,ecl_rate,ecl_increase_vs_base,ecl_increase_pct_vs_base
0,Base,1.00,1.0,750967975.0,1.462972e+08,0.457411,0.414735,0.194812,0.000000e+00,0.000000
1,Mild stress,1.25,1.1,750967975.0,2.007463e+08,0.570941,0.456208,0.267317,5.444901e+07,0.372181
2,Severe stress,1.50,1.2,750967975.0,2.558385e+08,0.670914,0.497682,0.340678,1.095412e+08,0.748758


## Business Recommendations

- Use Stage 3 and high score band views as first dashboard filters.
- Keep the base, mild stress, and severe stress scenario comparison prominent.
- Present ECL concentration by grade and purpose as business-facing portfolio diagnostics.
- Keep the limitations visible because this is a simplified analytics prototype.

## Company-specific Relevance

The project is most relevant for roles involving credit risk, portfolio analytics, risk reporting, data analytics, fintech lending, financial research, and model documentation.

## Resume Bullet Extraction

The strongest resume angle is that the project connects Python, credit risk modeling, transparent ECL assumptions, scenario analysis, and business-ready reporting.

## Interview Talking Points

- Why logistic regression was used as the baseline.
- How PD, LGD, and EAD connect to ECL.
- Why the staging logic is simplified.
- How model limitations were documented instead of hidden.
- How the outputs can feed a dashboard.

## Limitations

- This project uses a public LendingClub-style dataset, not bank production data.
- PD scores are from a baseline model and are not calibrated regulatory PDs.
- EAD, LGD, staging, and scenario assumptions are simplified.
- The Streamlit dashboard has not been built yet.

## Next Steps for Dashboard

In [7]:
stage_lookup = stage_summary.set_index("group_value")
dashboard_summary = pd.DataFrame([
    {
        "total_loans": total_loans,
        "total_exposure": total_exposure,
        "total_ecl": total_ecl,
        "ecl_rate": ecl_rate,
        "average_pd": average_pd,
        "average_lgd": average_lgd,
        "stage_1_count": int(stage_lookup.loc["Stage 1", "loan_count"]) if "Stage 1" in stage_lookup.index else 0,
        "stage_2_count": int(stage_lookup.loc["Stage 2", "loan_count"]) if "Stage 2" in stage_lookup.index else 0,
        "stage_3_count": int(stage_lookup.loc["Stage 3", "loan_count"]) if "Stage 3" in stage_lookup.index else 0,
        "stage_1_ecl": float(stage_lookup.loc["Stage 1", "total_ecl"]) if "Stage 1" in stage_lookup.index else 0.0,
        "stage_2_ecl": float(stage_lookup.loc["Stage 2", "total_ecl"]) if "Stage 2" in stage_lookup.index else 0.0,
        "stage_3_ecl": float(stage_lookup.loc["Stage 3", "total_ecl"]) if "Stage 3" in stage_lookup.index else 0.0,
    }
])

dashboard_summary.to_csv(PREDICTIONS_DIR / "dashboard_summary.csv", index=False)
stage_summary.to_csv(PREDICTIONS_DIR / "ecl_by_stage.csv", index=False)
score_band_summary.to_csv(PREDICTIONS_DIR / "ecl_by_score_band.csv", index=False)
if not grade_summary.empty:
    grade_summary.to_csv(PREDICTIONS_DIR / "ecl_by_grade.csv", index=False)
if not purpose_summary.empty:
    purpose_summary.to_csv(PREDICTIONS_DIR / "ecl_by_purpose.csv", index=False)

display(dashboard_summary)
print("Saved dashboard-ready CSV tables to outputs/predictions/.")

,total_loans,total_exposure,total_ecl,ecl_rate,average_pd,average_lgd,stage_1_count,stage_2_count,stage_3_count,stage_1_ecl,stage_2_ecl,stage_3_ecl
0,50000,750967975.0,1.462972e+08,0.194812,0.457411,0.414735,4995,20521,24484,3.819344e+06,4.215737e+07,1.003205e+08


Saved dashboard-ready CSV tables to outputs/predictions/.
